In [14]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px

# Add rebuild to path
sys.path.insert(0, str(Path.cwd() / "rebuild"))

# Set up output directories
OUTPUT_DIR = Path("./rebuild_outputs")
TABLES_DIR = OUTPUT_DIR / "tables"
FIGURES_DIR = OUTPUT_DIR / "figures"

TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Load results
df_results = pd.read_csv(TABLES_DIR / "results_all.csv")
df_summary = pd.read_csv(TABLES_DIR / "summary_statistics.csv")

print("✓ Loaded results from rebuild_outputs/")
print(f"  Total rows: {len(df_results)}")
print(f"  Models: {df_results['model'].unique().tolist()}")
print(f"  Mitigations: {df_results['mitigation'].unique().tolist()}")
print(f"  Perturbations: {df_results['perturbation'].unique().tolist()}")


✓ Loaded results from rebuild_outputs/
  Total rows: 36000
  Models: ['LogisticGLM', 'XGBoost', 'GRU']
  Mitigations: ['none', 'reweighting', 'smote', 'fairness_penalty']
  Perturbations: ['D0', 'D1A', 'D2A']


# Summary Statistics Table

# Pivot to get means for each combination
summary_pivot = df_summary.pivot_table(
    index=['model', 'mitigation', 'perturbation'],
    values=['utility_mean', 'disparate_impact_mean', 'equal_opportunity_mean'],
    aggfunc='first'
).reset_index()

summary_pivot = summary_pivot.sort_values(['model', 'perturbation', 'mitigation'])

# Create interactive table
fig = go.Figure(data=[go.Table(
    header=dict(
        values=['<b>Model</b>', '<b>Mitigation</b>', '<b>Perturbation</b>', 
                '<b>Utility</b>', '<b>Disparate Impact</b>', '<b>Equal Opportunity</b>'],
        fill_color='lightgrey',
        align='left',
        font=dict(size=12)
    ),
    cells=dict(
        values=[
            summary_pivot['model'],
            summary_pivot['mitigation'],
            summary_pivot['perturbation'],
            summary_pivot['utility_mean'].round(3),
            summary_pivot['disparate_impact_mean'].round(3),
            summary_pivot['equal_opportunity_mean'].round(3),
        ],
        align='left',
        height=30
    )
)])

fig.update_layout(
    title="Summary: Mean Metrics Across Bootstrap Iterations",
    height=1200
)

try:
    fig.write_image(str(FIGURES_DIR / "01_summary_table.png"), scale=2, width=1200, height=1200)
    print(f"✓ Saved: {FIGURES_DIR / '01_summary_table.png'}")
except Exception as e:
    print(f"⚠ Could not save PNG (kaleido required): {e}")
    print(f"  Install with: pip install kaleido")

fig.show()


In [27]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import numpy as np

# Heatmaps: Utility by Model, Mitigation, Perturbation
# Single figure with 3 subplots and shared colorbar

perturbation_map = {
    "D0": "Original",
    "D1A": "Row Removal",
    "D2A": "MAR"
}

mitigation_map = {
    "none": "None",
    "reweighting": "Reweighting",
    "smote": "SMOTE",
    "fairness_penalty": "Fairness Penalty"
}

perturbation_order = ["Original", "Row Removal", "MAR"]
mitigation_order = ["None", "Reweighting", "SMOTE", "Fairness Penalty"]

plot_df = df_summary.copy()

plot_df["perturbation"] = plot_df["perturbation"].replace(perturbation_map)
plot_df["mitigation"] = plot_df["mitigation"].replace(mitigation_map)

models = ["LogisticGLM", "GRU", "XGBoost"]

fig = make_subplots(
    rows=1,
    cols=3,
    subplot_titles=models,
    horizontal_spacing=0.08
)

for i, model_name in enumerate(models, start=1):
    df_model = plot_df[plot_df["model"] == model_name]

    pivot_util = df_model.pivot_table(
        index="perturbation",
        columns="mitigation",
        values="utility_mean",
        aggfunc="first"
    ).reindex(
        index=perturbation_order,
        columns=mitigation_order
    )

    fig.add_trace(
        go.Heatmap(
            z=pivot_util.values,
            x=pivot_util.columns,
            y=pivot_util.index,
            coloraxis="coloraxis",
            text=np.round(pivot_util.values, 3),
            texttemplate="%{text}",
            textfont=dict(size=11)
        ),
        row=1,
        col=i
    )

fig.update_layout(
    coloraxis=dict(
        colorscale="RdBu",
        cmin=0,
        cmax=1,
        colorbar=dict(title="Utility")
    ),
    height=400,
    width=1100
)

# Optional: avoid repeated y-axis labels
fig.update_yaxes(showticklabels=False, col=2)
fig.update_yaxes(showticklabels=False, col=3)

fname = "02_utility_heatmap_all_models.png"

try:
    fig.write_image(str(FIGURES_DIR / fname), scale=2, width=1100, height=400)
    print(f"✓ Saved: {FIGURES_DIR / fname}")
except Exception as e:
    print(f"⚠ Could not save {fname}: {e}")

fig.show()

✓ Saved: rebuild_outputs/figures/02_utility_heatmap_all_models.png


# Heatmaps: Disparate Impact by Model, Mitigation, Perturbation

for model_name in sorted(df_summary['model'].unique()):
    df_model = df_summary[df_summary['model'] == model_name]
    
    # Pivot: rows=perturbation, cols=mitigation, values=disparate_impact_mean
    pivot_di = df_model.pivot_table(
        index='perturbation',
        columns='mitigation',
        values='disparate_impact_mean',
        aggfunc='first'
    )
    
    # Ensure consistent ordering
    pivot_di = pivot_di.reindex(
        index=['D0', 'D1A', 'D2A'],
        columns=['none', 'reweighting', 'smote', 'threshold_optimization']
    )
    
    fig = px.imshow(
        pivot_di,
        labels=dict(x="Mitigation", y="Perturbation", color="Disparate Impact"),
        text_auto='.3f',
        aspect='auto',
        color_continuous_scale='RdYlGn',
        zmin=0.5, zmax=1.5
    )
    
    fig.update_layout(
        title=f"{model_name}: Disparate Impact P(alarm|F)/P(alarm|M) (ideal = 1.0)",
        height=400,
        width=700
    )
    
    fname = f"03_{model_name}_disparate_impact_heatmap.png"
    try:
        fig.write_image(str(FIGURES_DIR / fname), scale=2, width=700, height=400)
        print(f"✓ Saved: {FIGURES_DIR / fname}")
    except Exception as e:
        print(f"⚠ Could not save {fname}: {e}")
    
    fig.show()


In [18]:
df_results

,iteration,model,mitigation,perturbation,overall_physionet_utility,disparate_impact,equal_opportunity,n_alarms
0,1,LogisticGLM,none,D0,0.618645,1.066819,0.093153,14643
1,2,LogisticGLM,none,D0,0.662181,1.002854,0.007973,14142
2,3,LogisticGLM,none,D0,0.606497,1.037128,-0.044576,13807
3,4,LogisticGLM,none,D0,0.597800,0.956050,-0.015656,14646
4,5,LogisticGLM,none,D0,0.616421,1.018343,-0.059321,13624
...,...,...,...,...,...,...,...,...
35995,996,GRU,fairness_penalty,D2A,-0.000014,NaN,0.000000,1
35996,997,GRU,fairness_penalty,D2A,0.001440,NaN,0.002622,8
35997,998,GRU,fairness_penalty,D2A,0.000000,NaN,0.000000,0
35998,999,GRU,fairness_penalty,D2A,-0.000016,NaN,0.000000,2


In [19]:
import pandas as pd
import plotly.graph_objects as go

def make_model_table(df_results, model_name):

    dataset_map = {
        "D0": "Original",
        "D1A": "Row Removal",
        "D2A": "MAR"
    }

    mitigation_map = {
        "none": "None",
        "reweighting": "Reweighting",
        "smote": "SMOTE",
        "fairness_penalty": "Fairness Penalty"
    }

    dataset_order = ["Original", "Row Removal", "MAR"]
    mitigation_order = ["None", "Reweighting", "SMOTE", "Fairness Penalty"]

    df = df_results.copy()

    df["Dataset"] = df["perturbation"].replace(dataset_map)
    df["Mitigation"] = df["mitigation"].replace(mitigation_map)

    df = df[df["model"] == model_name]

    # summarize across iterations
    df = (
        df.groupby(["Dataset", "Mitigation"], as_index=False)
          .agg(
              Utility=("overall_physionet_utility", "mean"),
              **{
                  "Disparate Impact": ("disparate_impact", "mean"),
                  "Equal Opportunity": ("equal_opportunity", "mean")
              }
          )
          .round(3)
    )

    df["Dataset"] = pd.Categorical(df["Dataset"], categories=dataset_order, ordered=True)
    df["Mitigation"] = pd.Categorical(df["Mitigation"], categories=mitigation_order, ordered=True)

    df = df.sort_values(["Dataset", "Mitigation"])

    fig = go.Figure(data=[go.Table(
        header=dict(
            values=list(df.columns),
            fill_color="lightgrey",
            align="left",
            font=dict(size=16)
        ),
        cells=dict(
            values=[df[c] for c in df.columns],
            align="left",
            height=35
        )
    )])

    fig.update_layout(
        title=f"{model_name} — Dataset-first Comparison",
        height=600
    )

    print("\n", model_name)
    print(df.to_string(index=False))

    return fig


figs = {}

for m in ["LogisticGLM", "GRU", "XGBoost"]:
    figs[m] = make_model_table(df_results, m)

for f in figs.values():
    f.show()


 LogisticGLM
    Dataset       Mitigation  Utility  Disparate Impact  Equal Opportunity
   Original             None    0.630             1.036              0.012
   Original      Reweighting    0.627             1.036              0.013
   Original            SMOTE    0.624             1.037              0.011
   Original Fairness Penalty    0.002             4.227              0.003
Row Removal             None    0.616             1.036              0.011
Row Removal      Reweighting    0.618             1.038              0.010
Row Removal            SMOTE    0.620             1.040              0.012
Row Removal Fairness Penalty    0.001             3.302              0.002
        MAR             None    0.630             1.042              0.017
        MAR      Reweighting    0.627             1.041              0.016
        MAR            SMOTE    0.624             1.041              0.016
        MAR Fairness Penalty    0.001             4.148              0.002

 GRU
    D

In [26]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import numpy as np

perturbation_map = {
    "D0": "Original",
    "D1A": "Row Removal",
    "D2A": "MAR"
}

mitigation_map = {
    "none": "None",
    "reweighting": "Reweighting",
    "smote": "SMOTE",
    "fairness_penalty": "Fairness Penalty"
}

perturbation_order = ["Original", "Row Removal", "MAR"]
mitigation_order = ["None", "Reweighting", "SMOTE", "Fairness Penalty"]

plot_df = df_results.copy()
plot_df["perturbation"] = plot_df["perturbation"].replace(perturbation_map)
plot_df["mitigation"] = plot_df["mitigation"].replace(mitigation_map)

# --- compute mean EO ---
summary = (
    plot_df
    .groupby(["model", "perturbation", "mitigation"], as_index=False)
    .agg(eo_mean=("equal_opportunity", "mean"))
)

models = ["LogisticGLM", "GRU", "XGBoost"]

# --- determine shared color scale ---
all_vals = summary["eo_mean"].values
zmax = np.nanmax(np.abs(all_vals))

# --- create subplots ---
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=models,
    horizontal_spacing=0.15
)

for i, model_name in enumerate(models, start=1):
    df_model = summary[summary["model"] == model_name]

    pivot = df_model.pivot_table(
        index="perturbation",
        columns="mitigation",
        values="eo_mean",
        aggfunc="first"
    ).reindex(
        index=perturbation_order,
        columns=mitigation_order
    )

    fig.add_trace(
        go.Heatmap(
            z=pivot.values,
            x=pivot.columns,
            y=pivot.index,
            coloraxis="coloraxis",
            text=np.round(pivot.values, 3),
            texttemplate="%{text}",
            textfont={"size": 11}
        ),
        row=1, col=i
    )

# --- shared colorbar ---
fig.update_layout(
    coloraxis=dict(
        colorscale="RdBu_r",
        cmin=-zmax,
        cmax=zmax,
        colorbar=dict(title="Equal Opportunity")
    ),
    height=400,
    width=1100
)

fig.show()

In [23]:
# Box plots: Distribution of Utility Across Bootstrap Iterations

perturbation_map = {
    "D0": "Original",
    "D1A": "Row Removal",
    "D2A": "MAR"
}

mitigation_map = {
    "none": "None",
    "reweighting": "Reweighting",
    "smote": "SMOTE",
    "fairness_penalty": "Fairness Penalty"
}

perturbation_order = ["Original", "Row Removal", "MAR"]
mitigation_order = ["None", "Reweighting", "SMOTE", "Fairness Penalty"]

plot_df = df_results.copy()

plot_df["perturbation"] = plot_df["perturbation"].replace(perturbation_map)
plot_df["mitigation"] = plot_df["mitigation"].replace(mitigation_map)

for model_name in sorted(plot_df["model"].unique()):
    df_model = plot_df[plot_df["model"] == model_name]
    
    fig = px.box(
        df_model,
        x="mitigation",
        y="overall_physionet_utility",
        color="mitigation",
        facet_col="perturbation",
        category_orders={
            "mitigation": mitigation_order,
            "perturbation": perturbation_order
        },
        labels={
            "overall_physionet_utility": "Utility",
            "mitigation": "Mitigation",
            "perturbation": "Dataset"
        },
        title=f"{model_name}: Utility Distribution Across Bootstrap Iterations",
        height=500,
        width=1000
    )
    
    # cleaner look
    fig.update_layout(
        showlegend=False
    )
    
    fname = f"05_{model_name}_utility_boxplot.png"
    
    try:
        fig.write_image(str(FIGURES_DIR / fname), scale=2, width=1000, height=500)
        print(f"✓ Saved: {FIGURES_DIR / fname}")
    except Exception as e:
        print(f"⚠ Could not save {fname}: {e}")
    
    fig.show()


print("\n✓ All visualizations complete!")
print(f"Output directory: {FIGURES_DIR.resolve()}")

✓ Saved: rebuild_outputs/figures/05_GRU_utility_boxplot.png


✓ Saved: rebuild_outputs/figures/05_LogisticGLM_utility_boxplot.png


✓ Saved: rebuild_outputs/figures/05_XGBoost_utility_boxplot.png



✓ All visualizations complete!
Output directory: /Users/roseva1/Desktop/home/rose1838/SP26/PUBH8475/PUBH8475/Final/vibe_init/rebuild/rebuild_outputs/figures
